[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Pandas, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pandas-deep-dive.html)

# Reading Data &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

In [1]:
import math
import shutil
from datetime import datetime, timedelta
from pathlib import Path

import pandas as pd

STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}   # each station's mean for the year
LATITUDES = {"Bergen": 60.39, "Oslo": 59.91, "Svalbard": 78.22, "Tromso": 69.65, "Kirkenes": 69.73}


def year_of_readings():
    """Every hour of 2025 at the four stations, as the sqlite3 guide makes them, with Svalbard silent on 2 March."""
    for n in range(365 * 24):
        hour = datetime(2025, 1, 1) + timedelta(hours=n)
        season = -math.cos(2 * math.pi * (n - 400) / (365 * 24))
        day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
        for i, (station, mean) in enumerate(STATIONS.items()):
            if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
                celsius = None
            else:
                wobble = ((n * 37 + i * 101) % 17 - 8) / 10
                celsius = round(mean + 9 * season + 3 * day + wobble, 1) + 0.0
            yield station, hour.strftime("%Y-%m-%dT%H:%M"), celsius


readings = pd.DataFrame(list(year_of_readings()), columns=["station", "time", "celsius"])
readings["time"] = pd.to_datetime(readings["time"], format="%Y-%m-%dT%H:%M")
stations = pd.DataFrame({"station": list(LATITUDES), "latitude": list(LATITUDES.values())})

SCRATCH = Path("scratch")
SCRATCH.mkdir(exist_ok=True)
numbered = stations.assign(id=["0101", "0102", "0103", "0104", "0105"])[["id", "station", "latitude"]]
oslo_hours = readings[readings["station"] == "Oslo"].head(6)
svalbard_hours = readings[(readings["station"] == "Svalbard")
                          & readings["time"].between("2025-03-01 22:00", "2025-03-02 02:00")]

readings.to_csv(SCRATCH / "readings.csv", index=False)
numbered.to_csv(SCRATCH / "stations.csv", index=False)
numbered.to_csv(SCRATCH / "stations_excel.csv", index=False, encoding="utf-8-sig")
numbered.replace({"Tromso": "Tromsø"}).to_csv(SCRATCH / "stations_latin1.csv", index=False, encoding="latin-1")
oslo_hours.to_csv(SCRATCH / "oslo_export.csv", sep=";", decimal=",", index=False, date_format="%Y-%m-%d %H:%M")
svalbard_hours.to_csv(SCRATCH / "svalbard_export.csv", index=False, na_rep="--", date_format="%Y-%m-%d %H:%M")
(SCRATCH / "hand_edited.csv").write_text("station,time,celsius\n"
                                         "Bergen,2025-01-01 00:00,-3.6\n"
                                         "Oslo,2025-01-01 00:00,-3.5,checked\n"
                                         "Svalbard,2025-01-01 00:00,-14.6\n")

print("pandas", pd.__version__, "|", sorted(path.name for path in SCRATCH.iterdir()))

pandas 2.2.3 | ['hand_edited.csv', 'oslo_export.csv', 'readings.csv', 'stations.csv', 'stations_excel.csv', 'stations_latin1.csv', 'svalbard_export.csv']


**1.** Read `scratch/readings.csv` with `time` read as dates and times, and print the table's dtypes.

In [2]:
from_file = pd.read_csv(SCRATCH / "readings.csv", parse_dates=["time"])

print(from_file.dtypes)

station            object
time       datetime64[ns]
celsius           float64
dtype: object


Without `parse_dates`, `time` is text, `object`, as the first worked example showed.

**2.** Read only the `station` and `celsius` columns of the first 12 rows of `scratch/readings.csv`, and print them.

In [3]:
print(pd.read_csv(SCRATCH / "readings.csv", usecols=["station", "celsius"], nrows=12))

     station  celsius
0     Bergen     -3.6
1       Oslo     -3.5
2   Svalbard    -14.6
3     Tromso     -6.7
4     Bergen     -3.7
5       Oslo     -5.3
6   Svalbard    -16.4
7     Tromso     -8.5
8     Bergen     -3.7
9       Oslo     -5.3
10  Svalbard    -16.4
11    Tromso     -8.5


The first 12 rows are the year's first three hours, four stations each.

**3.** Read `scratch/oslo_export.csv` so that `celsius` comes out as numbers, and print Oslo's lowest reading in it.

In [4]:
oslo = pd.read_csv(SCRATCH / "oslo_export.csv", sep=";", decimal=",")

print(oslo["celsius"].min())

-5.3


`sep` splits the lines at the semicolons and `decimal` reads the commas as decimal points.

**4.** Read `scratch/stations.csv` with the ids kept as text and used as the index, and print the station whose id is `0103`.

In [5]:
numbered = pd.read_csv(SCRATCH / "stations.csv", dtype={"id": str}, index_col="id")

print(numbered.loc["0103"])

station     Svalbard
latitude       78.22
Name: 0103, dtype: object


With the ids as text, `0103` is a label of the index, written exactly as the file wrote it.

**5.** Read `scratch/svalbard_export.csv` so that `--` counts as missing, and print how many of its readings are missing.

In [6]:
svalbard = pd.read_csv(SCRATCH / "svalbard_export.csv", na_values="--")

print(svalbard["celsius"].isna().sum())

3


Three of the five hours, midnight to 02:00 on 2 March, are `--` in the file and `NaN` in the table.

The answers are finished with the files, so this cell removes the `scratch` folder.

In [7]:
shutil.rmtree(SCRATCH)

print("scratch still there:", SCRATCH.exists())

scratch still there: False


---

&#8592; **Back to:** [Reading Data](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pandas-deep-dive/04-reading-data.ipynb)  &nbsp;&middot;&nbsp;  [Pandas, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pandas-deep-dive.html)
